In [19]:
#| default_exp frida

In [1]:
#| hide
import nbdev; nbdev.nbdev_export()

/usr/local/lib/python3.12/dist-packages/nbdev/export.py:88: UserWarning: Notebook '/workspaces/gpt/rugptxl_converter.ipynb' uses `#|export` without `#|default_exp` cell.
Note nbdev2 no longer supports nbdev1 syntax. Run `nbdev_migrate` to upgrade.
See https://nbdev.fast.ai/getting_started.html for more information.
  warn(f"Notebook '{nbname}' uses `#|export` without `#|default_exp` cell.\n"


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [3]:
#| export
from os import getenv
model_path = getenv("MODEL")

In [4]:
model_path = 'fred'

In [5]:
#| export
from optimum.onnxruntime import ORTModelForSeq2SeqLM

In [ ]:
#| export
seq_length = 1024
seq_length = 512

full_path = f'./models/{model_path}'
import torch
from transformers import GPT2Tokenizer, T5ForConditionalGeneration, AutoTokenizer

In [7]:
def convert_and_save_model(model_path, save_dir):
    model = ORTModelForSeq2SeqLM.from_pretrained(model_path, export=True)
    model.save_pretrained(save_dir)
    
    # Also save the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    tokenizer.save_pretrained(save_dir)
    
    print(f"Model and tokenizer saved to {save_dir}")

In [8]:
#convert_and_save_model(full_path, full_path+'/optimized')

In [9]:
#| export
tokenizer = GPT2Tokenizer.from_pretrained(full_path+'/optimized/', eos_token='</s>')
model = ORTModelForSeq2SeqLM.from_pretrained(full_path+'/optimized/', provider="CUDAExecutionProvider")

2025-04-28 06:03:08.532061827 [W:onnxruntime:, session_state.cc:1263 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-04-28 06:03:08.532086227 [W:onnxruntime:, session_state.cc:1265 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2025-04-28 06:03:15.239609841 [W:onnxruntime:, session_state.cc:1263 VerifyEachNodeIsAssignedToAnEp] Some nodes were not assigned to the preferred execution providers which may or may not have an negative impact on performance. e.g. ORT explicitly assigns shape related ops to CPU to improve perf.
2025-04-28 06:03:15.239625281 [W:onnxruntime:, session_state.cc:1265 VerifyEachNodeIsAssignedToAnEp] Rerunning with verbose output on a non-minimal build will show node assignments.
2025-04-28 06:03:24.255317371 [W:onnxrun

In [10]:
tokenizer.model_max_length

1000000000000000019884624838656

In [11]:
#| export

from rest.gen import get_line_enders, iftoken

def iftoken(tokenizer, tokens):
    # returns token id if the given string is one token
    token_ids = [tokenizer.encode(token, add_special_tokens=False) for token in tokens]
    return [id for sublist in token_ids for id in sublist if len(sublist) == 1]

In [12]:
#| export
from front.common import process_seq

line_enders = get_line_enders(tokenizer)

def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool, temperature:float=0.1):
    max_input = 2*1024 - length
    prompt = tokenizer.decode(tokenizer.encode(prompt)[-max_input:]).removeprefix('<|begin_of_text|>')

    blocked_tokens = ["».",'.\n','\n\t\t','http://',',[','("','.]',' («',')','\u2004',']','(«','[', ' [', '(', ' (', '\xa0', '*', '­', '~', '_', '\\', '\uf04a', '\ufeff', '\u2028']
    if not allow_linebreak:
        blocked_tokens.extend(['\n', '\n\n',' \n'])
    bad_words_ids = iftoken(tokenizer, blocked_tokens)
    bad_words_ids = [[w] for w in bad_words_ids]
    
    lm_text = '<LM>' + prompt
    input_ids=torch.tensor([tokenizer.encode(lm_text)]).cuda()
    torch.cuda.empty_cache()
    output_ids = model.generate(input_ids, do_sample=True, temperature=temperature, repetition_penalty=5.0, min_p=0.1, #watermark=False, typical_p=0.9, top_k=10, top_p=0.95,
                        max_new_tokens=length, bad_words_ids = bad_words_ids,
                        num_return_sequences=num_samples,)
    
    result = [tokenizer.decode(o[1:]).replace('\n', ' ') for o in output_ids]
    result = process_seq(result)
    return result


In [13]:
length = 50

In [14]:
max_input = 1024*2 - length
prompt = '<LM>'+'На словах ты Лев Толстой, а на деле'*100000
prompt = tokenizer.decode(tokenizer.encode(prompt)[-max_input:]).removeprefix('<|begin_of_text|>')


In [15]:
%%time
get_sample(prompt, length, 4, False)

CPU times: user 2.45 s, sys: 1.08 s, total: 3.53 s
Wall time: 2.99 s


[', а на делеНа словах ты Лев Толстой. А потом я понял: это не так уж и важно – кто он такой по жизни? Важно то же самое для меня самого!',
 ', а на делеНа словах ты Лев Толстой. А потом я понял: это не так уж и важно – кто говорит правду о себе самом? Важно то же самое сказать другим людям… И тогда все становится ясно!',
 ', а на делеНа словах ты Лев Толстой… На этих строках я остановился.',
 ', а на делеНа словах ты Лев Толстой… На этих строках я остановился. Я не знал точно – что именно мне надо сказать дальше и как это сделать правильно? И вообще: кто такой этот «я»?']

In [16]:
%%time
get_sample('<LM>На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.19 s, sys: 115 ms, total: 1.3 s
Wall time: 724 ms


[' – просто говно. И не надо мне тут про «венец творения» и все такое прочее, я это уже слышал от тебя тысячу раз!',
 ' – просто говно. И не надо мне тут про «свободу слова» и все такое прочее… Я, может быть тоже хочу свободы!',
 ' – просто говно. И не надо мне тут про «свободу слова» и все такое прочее… Я, между прочим тоже могу сказать тебе пару ласковых слов по поводу твоего творчества!',
 ' – просто говно. И не надо мне тут про «свободу слова» и все такое прочее… Ты, главное дело — помни: ты сам себе хозяин! Понял? Сам!..']

In [17]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.15 s, sys: 0 ns, total: 1.15 s
Wall time: 568 ms


[' – просто говно.',
 ' – просто говно. И не надо мне тут про «свободу слова» и все такое прочее… Я, может быть тоже хочу свободы! Но я ее вижу в том числе по телевизору!',
 ' – говно. И не надо мне тут про «свободу слова» и все такое прочее, я знаю…» Но в этот момент он понял: если бы его спросили сейчас о том же самом — что это за свобода такая?',
 ' – просто говно. И не надо мне тут про «свободу слова» и прочую хрень, которую ты сам придумал для того чтобы оправдать свою никчемность… Ты даже в туалет без меня сходить боишься!']

In [18]:
%%time
get_sample('На словах ты Лев Толстой, а на деле', 50, 4, False)

CPU times: user 1.15 s, sys: 0 ns, total: 1.15 s
Wall time: 570 ms


[' – просто говно. И не надо мне тут про «свободу слова» и все такое прочее… Ты, главное дело помни: ты в России живешь! А Россия для русских!',
 ' – просто говно.',
 ' – просто говно.',
 ' – просто говно. И не надо мне тут про «свободу слова» и все такое прочее… Я, может быть тоже хочу свободы!']